# ACN-Data Calibration

Reproducible calibration run with diagnostic plots.

Requires `ACN_API_TOKEN` in env. Re-runs use the cache in `data/calibration/acn_cache/`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

REPO = Path('..').resolve()
POPS = REPO / 'configs' / 'populations.yaml'
CACHE = REPO / 'data' / 'calibration' / 'acn_cache'
ARTIFACTS = REPO / 'data' / 'calibration'
assert os.environ.get('ACN_API_TOKEN'), 'set ACN_API_TOKEN before running'

## 1. Run calibration

In [ ]:
from v2b_syndata.calibration import calibrate_populations

summary = calibrate_populations(
    populations_yaml_path=POPS,
    population_names=['consent_default'],
    sites=('caltech', 'jpl', 'office001'),
    year_start=2019,
    year_end=2021,
    cache_dir=CACHE,
    artifact_dir=ARTIFACTS,
)
summary

## 2. Load fitted populations.yaml

In [ ]:
with POPS.open() as f:
    pops = yaml.safe_load(f)
pop = pops['consent_default']
rd = pop.get('region_distributions', {})
meta = pop.get('calibration_metadata', {})
print('source:', meta.get('source'))
print('regions:', list(rd))
print('n_users:', meta.get('n_users_total'))
print('fallback_rate:', meta.get('capacity_inference_fallback_rate'))

## 3. KS-fit-quality table

In [ ]:
rows = []
for region, dists in rd.items():
    row = {'region': region}
    for d in ('arrival', 'dwell', 'soc_arrival'):
        if d in dists and 'ks_fit_quality' in dists[d]:
            row[f'{d}_ks'] = dists[d]['ks_fit_quality']
            row[f'{d}_n'] = dists[d].get('n_samples')
    rows.append(row)
ks_df = pd.DataFrame(rows)
ks_df

## 4. Per-user (φ, κ) scatter with region overlays

In [ ]:
users = pd.read_csv(ARTIFACTS / 'acn_per_user.csv')
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(users['phi'], users['kappa'], s=10, alpha=0.4)
for region in pop['axes_distribution']:
    f = region['freq']
    k = region['consist']
    ax.add_patch(plt.Rectangle((f[0], k[0]), f[1] - f[0], k[1] - k[0],
                                fill=False, edgecolor='C1'))
    ax.text(f[0], k[1], region['name'], fontsize=8, color='C1')
ax.set_xlabel('phi (frequency)')
ax.set_ylabel('kappa (consistency)')
ax.set_title('Per-user (phi, kappa) with axes_distribution regions')
plt.show()

## 5. Per-region fitted overlays

For each region with fitted parameters, plot empirical histogram of arrival/dwell/soc with the fitted CDF/PDF overlay.

In [ ]:
from scipy import stats as st

for region_name, dists in rd.items():
    if 'arrival' not in dists:
        continue
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    arr = dists['arrival']
    a, b = (6.0 - arr['mu']) / arr['sigma'], (20.0 - arr['mu']) / arr['sigma']
    x = np.linspace(6, 20, 200)
    axes[0].plot(x, st.truncnorm.pdf(x, a, b, loc=arr['mu'], scale=arr['sigma']),
                 label=f'fit μ={arr["mu"]:.2f} σ={arr["sigma"]:.2f}')
    axes[0].set_title(f'{region_name} — arrival_hour')
    axes[0].legend()
    if 'dwell' in dists:
        dw = dists['dwell']
        x = np.linspace(0.1, 24, 200)
        axes[1].plot(x, st.weibull_min.pdf(x, dw['k'], scale=dw['lambda']),
                     label=f'fit k={dw["k"]:.2f} λ={dw["lambda"]:.2f}')
        axes[1].set_title(f'{region_name} — dwell_hours')
        axes[1].legend()
    if 'soc_arrival' in dists:
        sa = dists['soc_arrival']
        x = np.linspace(0.001, 0.999, 200)
        axes[2].plot(x, st.beta.pdf(x, sa['alpha'], sa['beta']),
                     label=f'fit α={sa["alpha"]:.2f} β={sa["beta"]:.2f}')
        axes[2].set_title(f'{region_name} — arrival_soc')
        axes[2].legend()
    plt.tight_layout()
    plt.show()

## 6. Capacity-inference sensitivity sweep

Re-fit Beta(soc) under fixed-capacity assumptions {40, 60, 75, 100} kWh; report the parameter range to characterize sensitivity to the inference heuristic. Documented in CALIBRATION_NOTES.md.

In [ ]:
from v2b_syndata.calibration.acn_fetcher import (
    fetch_all_sessions, filter_with_userid,
)
from v2b_syndata.calibration.feature_extractor import extract_session
from v2b_syndata.calibration.distribution_fitter import fit_beta_soc

cap_grid = [40.0, 60.0, 75.0, 100.0]
site_to_sessions = {}
for site in ('caltech', 'jpl', 'office001'):
    raw = fetch_all_sessions(site, 2019, 2021, cache_dir=CACHE)
    raw = filter_with_userid(raw)
    site_to_sessions[site] = [s for s in (extract_session(r, site) for r in raw) if s is not None]
all_sess = [s for v in site_to_sessions.values() for s in v]

results = []
for cap in cap_grid:
    socs = [max(0, min(1, 1 - s.kwh_requested / cap)) for s in all_sess if s.kwh_requested]
    fit = fit_beta_soc(np.asarray(socs))
    results.append({'capacity_kwh': cap, **fit})
pd.DataFrame(results)